# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU**.

1. Smoke (`stability_smoke.yaml`) — minutes, already on GitHub.
2. **Paper matrix** (`stability.yaml`, `--profile t4`) — 34 cells × 800 items. Hours; resume-safe.

Use a **separate** registry (`results/registry_paper.jsonl`) so the n=4 smoke does not mix into rankings.

In [1]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"

Cloning into 'apertus-eval-prep'...
remote: Enumerating objects: 215, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 215 (delta 84), reused 195 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (215/215), 345.63 KiB | 8.23 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/apertus-eval-prep
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))
if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

Tesla T4
official slices already on disk


## Paper matrix (T4)

34 cells. `--profile t4` skips 7B fp16, 7B int8, and 7B vLLM.
Re-run this cell after a disconnect; finished `config_hash` rows are skipped.

To finish across sessions, run **one model at a time** (uncomment one `--only-model` line).

In [ ]:
# All 34 T4 cells. Prefer this if the runtime will stay up for many hours.
!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml \
  --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl

# If Colab dies, rerun the same command. Or do one model per session:
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model HuggingFaceTB/SmolLM2-1.7B-Instruct
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model Qwen/Qwen2.5-3B-Instruct
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model microsoft/Phi-3.5-mini-instruct
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model Qwen/Qwen2.5-7B-Instruct

run  SmolLM2-1.7B-Instruct_control_control_24ffe98d9250761d factor=control=control
config.json: 100% 908/908 [00:00<00:00, 4.05MB/s]
tokenizer_config.json: 100% 3.76k/3.76k [00:00<00:00, 14.0MB/s]
vocab.json: 100% 801k/801k [00:00<00:00, 22.4MB/s]
merges.txt: 100% 466k/466k [00:00<00:00, 103MB/s]
special_tokens_map.json: 100% 655/655 [00:00<00:00, 3.76MB/s]
tokenizer.json: 100% 2.10M/2.10M [00:00<00:00, 126MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.42G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   9% 306M/3.42G [00:01<00:10, 285MB/s, 27.0MB/s  ]
model.safetensors: downloading bytes:  11% 375M/3.42G [00:02<00:13, 219MB/s, 31.4MB/s  ]
model.safetensors: downloading bytes:  14% 486M/3.42G [00:02<00:13, 222MB/s, 40.7MB/s  ]
model.safetensors: downloading bytes:  23% 787M/3.42G [00:04<00:19, 138MB/s, 61.1MB/s  ]
model.safetensors: downloading bytes:  25% 842M/3.42G [00:04<00:19, 134MB/s, 63.2MB/s  ]
mod

In [ ]:
from google.colab import files
from pathlib import Path

!python -m apertus_eval_prep report --registry results/registry_paper.jsonl --out reports/stability_paper
!python -m apertus_eval_prep paper-tables --registry results/registry_paper.jsonl --out paper/_generated_tables.md
!zip -r paper_matrix_artifacts.zip results/runs results/registry_paper.jsonl reports/stability_paper paper/_generated_tables.md
print("zip bytes", Path("paper_matrix_artifacts.zip").stat().st_size)
files.download("paper_matrix_artifacts.zip")

Unpack `paper_matrix_artifacts.zip` into the Mac clone. Do not edit numbers. Commit `results/registry_paper.jsonl`, new `results/runs/*.json`, and `reports/stability_paper/`.